# Browser-Based Real-Time Face Swap with Colab GPU

This notebook captures your webcam directly in the browser, processes frames on Colab's GPU, and displays the face-swapped result in real-time.

**No Windows client needed!** Everything runs in this notebook.

## Steps:
1. Select a GPU runtime (T4 recommended)
2. Install dependencies
3. Upload a source face image
4. Start webcam capture and processing

## 1. Install Dependencies

In [ ]:
%%capture
!pip install huggingface_hub

!pip install onnxruntime-gpu==1.20.1 #--extra-index-url https://aiinfra.pkgs.visualstudio.com/PublicPackages/_packaging/onnxruntime-cuda-12/pypi/simple/
!pip install insightface #==0.7.3


## 2. Verify GPU

In [ ]:
import torch
import onnxruntime as ort

print("CUDA available:", torch.cuda.is_available())
print("ONNX Runtime providers:", ort.get_available_providers())

assert "CUDAExecutionProvider" in ort.get_available_providers(), "GPU not available!"


## 3. Download Face Swap Model

In [ ]:
import os
from pathlib import Path
from huggingface_hub import hf_hub_download
from google.colab import userdata

MODEL_DIR = Path("/content/models")
MODEL_DIR.mkdir(parents=True, exist_ok=True)

HF_REPO_ID = "ninjawick/webui-faceswap-unlocked"
HF_FILENAME = "inswapper_128.onnx"
SWAPPER_PATH = MODEL_DIR / HF_FILENAME

if not SWAPPER_PATH.exists():
    print(f"Downloading {HF_FILENAME} to {MODEL_DIR}...")
    try:
        # Get token if available
        try:
            hf_token = userdata.get('HF_TOKEN')
        except:
            hf_token = None

        # Download directly to the local directory
        downloaded_path = hf_hub_download(
            repo_id=HF_REPO_ID,
            filename=HF_FILENAME,
            local_dir=str(MODEL_DIR),
            token=hf_token
        )

        print(f"Successfully downloaded to: {downloaded_path}")
        print(f"File size: {os.path.getsize(SWAPPER_PATH) / 1024 / 1024:.1f} MB")
    except Exception as e:
        print(f"Error downloading model: {e}")
        print("If this is a private repo, ensure your HF_TOKEN is set in Colab secrets.")
else:
    print(f"Model already exists at: {SWAPPER_PATH}")


## 4. Initialize Face Processing

In [ ]:
import cv2
import insightface
import numpy as np

ORT_PROVIDERS = ["CUDAExecutionProvider", "CPUExecutionProvider"]

# Initialize face analyzer
print("Initializing face analyzer...")
FACE_ANALYSER = insightface.app.FaceAnalysis(name="buffalo_l", providers=ORT_PROVIDERS)
FACE_ANALYSER.prepare(ctx_id=0, det_size=(640, 640))

# Initialize face swapper
print("Initializing face swapper...")
FACE_SWAPPER = insightface.model_zoo.get_model(str(SWAPPER_PATH), providers=ORT_PROVIDERS)

print("✓ GPU face processing initialized")

def get_one_face(frame):
    """Detect the first face in the frame."""
    faces = FACE_ANALYSER.get(frame)
    if not faces:
        return None
    return min(faces, key=lambda face: face.bbox[0])

def swap_face(source_face, target_frame):
    """Swap source face into target frame."""
    target_face = get_one_face(target_frame)
    if target_face is None:
        return target_frame  # No face detected, return original
    
    result = FACE_SWAPPER.get(target_frame, target_face, source_face, paste_back=True)
    return result


## 5. Upload Source Face Image

Upload an image containing the face you want to swap onto your webcam.

In [ ]:
from google.colab import files
from IPython.display import Image, display
import io

print("Upload a source face image (JPG, PNG):")
uploaded = files.upload()

# Get the uploaded file
source_filename = list(uploaded.keys())[0]
source_bytes = uploaded[source_filename]

# Decode image
source_array = np.frombuffer(source_bytes, np.uint8)
SOURCE_IMAGE = cv2.imdecode(source_array, cv2.IMREAD_COLOR)

print(f"Source image loaded: {SOURCE_IMAGE.shape}")
display(Image(data=source_bytes, width=300))

# Extract source face
print("\nDetecting source face...")
SOURCE_FACE = get_one_face(SOURCE_IMAGE)
if SOURCE_FACE is None:
    raise ValueError("No face detected in source image!")
print("✓ Source face detected and cached")


## 6. Start Real-Time Face Swap

This will:
1. Capture your webcam in the browser
2. Send frames to Colab GPU for processing
3. Display the face-swapped result in real-time

**Click "Allow" when prompted for webcam access.**

Press **Stop** button to end the stream.

In [ ]:
from IPython.display import display, Javascript, HTML
from google.colab.output import eval_js
from base64 import b64decode, b64encode
import PIL.Image
import io
import time

def webcam_to_numpy(quality=0.8, size=(640, 480)):
    """Capture a frame from webcam and return as numpy array."""
    js = Javascript('''
    async function captureFrame(quality, width, height) {
      const video = document.createElement('video');
      const stream = await navigator.mediaDevices.getUserMedia({video: {width, height}});
      
      video.srcObject = stream;
      await video.play();
      
      // Wait for video to be ready
      await new Promise(resolve => setTimeout(resolve, 500));
      
      const canvas = document.createElement('canvas');
      canvas.width = video.videoWidth;
      canvas.height = video.videoHeight;
      canvas.getContext('2d').drawImage(video, 0, 0);
      
      stream.getTracks().forEach(track => track.stop());
      
      return canvas.toDataURL('image/jpeg', quality);
    }
    ''')
    display(js)
    
    data = eval_js(f'captureFrame({quality}, {size[0]}, {size[1]})')
    binary = b64decode(data.split(',')[1])
    
    img = PIL.Image.open(io.BytesIO(binary))
    return cv2.cvtColor(np.array(img), cv2.COLOR_RGB2BGR)

print("Starting webcam... (this may take a moment)")
print("="*50)

# Test capture
test_frame = webcam_to_numpy(quality=0.8, size=(640, 480))
print(f"✓ Webcam ready! Resolution: {test_frame.shape[1]}x{test_frame.shape[0]}")
print("\nProcessing first frame...")

# Process and display
result = swap_face(SOURCE_FACE, test_frame)
result_rgb = cv2.cvtColor(result, cv2.COLOR_BGR2RGB)
result_pil = PIL.Image.fromarray(result_rgb)

display(result_pil)
print("\n✓ Face swap working!")
print("\nRun the next cell for continuous streaming.")


## 7. Continuous Stream (Run this for live video)

**Note:** Due to Colab limitations, this captures frames repeatedly rather than true video streaming. Expect 2-5 FPS.

Press **Interrupt** (⏹️) to stop.

In [ ]:
from IPython.display import display, Javascript, HTML, clear_output
from google.colab.output import eval_js
from base64 import b64decode, b64encode
import numpy as np
import cv2
import PIL.Image
import io
import time

# Optimized JS with persistence. We use a unique ID to avoid duplicates.
JS_PERSISTENT = """
(function() {
  if (window.swapperInitialized) return;

  window.startWebcam = async function(width, height) {
    if (window.stream) { window.stopWebcam(); }
    window.video = document.createElement('video');
    window.video.style.display = 'none';
    window.video.width = width;
    window.video.height = height;

    window.stream = await navigator.mediaDevices.getUserMedia({video: {width: width, height: height}});
    window.video.srcObject = window.stream;
    await window.video.play();

    window.canvas = document.createElement('canvas');
    window.canvas.width = window.video.videoWidth;
    window.canvas.height = window.video.videoHeight;
    return [window.video.videoWidth, window.video.videoHeight];
  };

  window.captureFrame = async function(quality) {
    if (!window.canvas) return null;
    window.canvas.getContext('2d').drawImage(window.video, 0, 0);
    return window.canvas.toDataURL('image/jpeg', quality);
  };

  window.stopWebcam = function() {
    if (window.stream) {
      window.stream.getTracks().forEach(track => track.stop());
      if (window.video) window.video.remove();
      window.stream = null;
    }
  };

  window.swapperInitialized = true;
})();
"""

# Clear previous output but ensure JS is injected in a way that survives
display(Javascript(JS_PERSISTENT))

try:
    width, height = 480, 360
    eval_js(f'window.startWebcam({width}, {height})')
    time.sleep(2)

    # Use a fixed display handle to avoid clearing the whole cell output (including JS)
    display_handle = display(HTML("Initializing display..."), display_id=True)

    print("Streaming started. Press Stop (■) to end.")

    frame_count = 0
    start_time = time.time()

    while True:
        data_url = eval_js('window.captureFrame(0.4)')
        if not data_url:
            continue

        binary = b64decode(data_url.split(',')[1])
        img_array = np.frombuffer(binary, dtype=np.uint8)
        frame = cv2.imdecode(img_array, cv2.IMREAD_COLOR)

        if frame is None: continue

        # Process frame
        result = swap_face(SOURCE_FACE, frame)

        # Encode
        _, buffer = cv2.imencode('.jpg', result, [cv2.IMWRITE_JPEG_QUALITY, 70])
        encoded = b64encode(buffer).decode('utf-8')

        # Update existing display handle instead of clear_output
        fps = (frame_count + 1) / (time.time() - start_time)
        html_content = f'''
        <div style="font-family: monospace;">FPS: {fps:.1f} | Frames: {frame_count}</div>
        <img src="data:image/jpeg;base64,{encoded}" style="width: {width}px;">
        '''
        display_handle.update(HTML(html_content))

        frame_count += 1

except KeyboardInterrupt:
    print("
Stopping...")
except Exception as e:
    print(f"
Error: {e}")
finally:
    try:
        eval_js('window.stopWebcam()')
    except: pass
    print("Webcam released.")


## Notes

- **Performance**: Browser → Colab → Browser adds latency. Expect 2-5 FPS.
- **Privacy**: Your webcam data stays in your browser and Colab session. Nothing is stored.
- **GPU**: Uses Colab's T4 GPU for real-time face swapping.
- **Limitations**: This won't work in Zoom/Teams directly - it's just for preview.

For production use with virtual cameras, use the Windows client version instead.